# 04 - Büyük veri: ikinci veri seti ile model

Bu defter, Kaggle yarışma verisine (784 ilan) **ikinci, çok büyük bir veri seti** eklemenin
modele etkisini ölçer.

1. İki veri setinin tanıtımı
2. Çöp ilan temizliği (kural kural, iki set için)
3. Sızıntı kontrolü: aynı ilan iki sette de var mı?
4. A / B / C / D karşılaştırması (her zaman aynı temiz Kaggle ilanlarında ölçülür)
5. Yeni modelle özellik önemi

Çalışma süresi birkaç on dakika sürebilir (C ve D, 5'er kez 170 bin satırla eğitilir).

In [1]:
import os, sys, time, warnings
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, train_test_split

from src.db.load_data import VEHICLES_COLS
from src.ml import models
from src.ml.config import (CAT_COLS, FEATURES, KAGGLE_WEIGHT, RAW_TRAIN_PATH, RAW_VEHICLES_PATH,
                           TARGET)
from src.ml.models import BEST_PARAMS, make_big_hist_gb, make_hist_gb
from src.ml.preprocess import clean, filter_report, filter_train_rows

os.makedirs("docs/img", exist_ok=True)
pd.options.display.float_format = "{:,.0f}".format

## 1. İki veri seti

| | Kaggle (yarışma) | vehicles.csv |
|---|---|---|
| Kaynak | Craigslist ABD, haziran 2026 | Craigslist ABD, eylül-kasım 2026 |
| Boyut | 784 ilan | yaklaşık 295 bin ilan |
| Satıcı tipi | yok (metinden tahmin ederiz) | var (`dealer` / `owner`) |
| Rolü | **asıl hedef**: model burada ölçülür | eğitime ek veri |

Kaggle verisi çok küçük; birçok marka-model için birkaç ilan var. vehicles.csv aynı türden ilanlar
ama yüzlerce kat fazla: model "Camry 2015" gibi örüntüleri çok daha iyi öğrenebilir.

In [2]:
kaggle_raw = pd.read_csv(RAW_TRAIN_PATH)
vehicles_raw = pd.read_csv(RAW_VEHICLES_PATH, usecols=VEHICLES_COLS, low_memory=False)
kaggle_clean = clean(kaggle_raw)
vehicles_clean = clean(vehicles_raw.rename(columns={"id": "ID"}))
print("Kaggle:", len(kaggle_clean), "ilan   vehicles:", len(vehicles_clean), "ilan")
print()
print("Medyan fiyat ($):  Kaggle", int(kaggle_clean[TARGET].median()),
      "  vehicles", int(vehicles_clean[TARGET].median()))
print("Galeri oranı:      Kaggle (metinden)", round((kaggle_clean.seller_type == "Galeri").mean(), 3),
      "  vehicles (gerçek etiket)", round((vehicles_clean.seller_type == "Galeri").mean(), 3))

Kaggle: 784 ilan   vehicles: 295451 ilan

Medyan fiyat ($):  Kaggle 9500   vehicles 12500
Galeri oranı:      Kaggle (metinden) 0.46   vehicles (gerçek etiket) 0.739


## 2. Çöp ilan temizliği

Ham ilanlarda araç olmayanlar (bisiklet, ekskavatör), peşinat ilanları ("1 $"), fiyatı fazladan
sıfırlı olanlar, sayacı bozuk olanlar ve aynı ilanın tekrarları var. `filter_train_rows()` bunları
sabit ve açıklanabilir kurallarla atar. Aşağıdaki tablo her kuralın, **öncekilerden kalan**
satırlardan kaç tanesini attığını gösterir (sırayla uygulanır).

In [3]:
rep_k = dict(filter_report(kaggle_clean))
rep_v = dict(filter_report(vehicles_clean))
table = pd.DataFrame({"Kaggle": rep_k, "vehicles": rep_v})
table.loc["TOPLAM atılan"] = table.sum()
table.loc["Başlangıç"] = [len(kaggle_clean), len(vehicles_clean)]
table.loc["Kalan"] = table.loc["Başlangıç"] - table.loc["TOPLAM atılan"]
table

,Kaggle,vehicles
marka bulunamayan (araç olmayan / reklam),40,4453
"fiyat 1,000 $ altı (peşinat / 'ara' ilanı)",6,14409
"fiyat 250,000 $ üstü (hatalı)",0,212
yıl 1950 öncesi veya yok,62,903
"km yok, 600,000 km üstü veya bozuk sayaç",13,7620
2.000 km altı ama 2024 öncesi (sahte km),10,2029
'Parça için' ruhsat,1,55
"tekrar ilan (marka, model, yıl, km, fiyat aynı)",0,98042
TOPLAM atılan,132,127723
Başlangıç,784,295451


In [4]:
kaggle = filter_train_rows(kaggle_clean).reset_index(drop=True)
vehicles_all = filter_train_rows(vehicles_clean).reset_index(drop=True)
print(len(kaggle), "temiz Kaggle ilanı;", len(vehicles_all), "temiz vehicles ilanı")

652 temiz Kaggle ilanı; 167728 temiz vehicles ilanı


## 3. Sızıntı kontrolü

Aynı ilan hem Kaggle'da hem vehicles.csv'de olabilir (ikisi de Craigslist). Model Kaggle ilanlarında
ölçüldüğü için, bu ilanlar eğitim setinde kalırsa model **cevabı önceden görmüş olur** ve sonuç
olduğundan iyi çıkar. (Başlık, fiyat) çifti aynı olan ilanları vehicles'tan çıkarırız.

In [5]:
pairs = kaggle_clean[["title", TARGET]].drop_duplicates()
merged = vehicles_all.merge(pairs, on=["title", TARGET], how="left", indicator=True)
leak = int((merged["_merge"] == "both").sum())
vehicles = vehicles_all[(merged["_merge"] == "left_only").to_numpy()].reset_index(drop=True)
print(f"Kaggle ile çakışan ve çıkarılan vehicles ilanı: {leak}")
print(f"Eğitimde kalan vehicles ilanı: {len(vehicles)}")

Kaggle ile çakışan ve çıkarılan vehicles ilanı: 17
Eğitimde kalan vehicles ilanı: 167711


## 4. A / B / C / D karşılaştırması

Ölçüm her seferinde **aynı 652 temiz Kaggle ilanı** üzerinde, 5 katlı çapraz doğrulamayla yapılır.
Vehicles ilanları yalnızca eğitime girer; ölçülen ilanlar modelin hiç görmediği katlardır.

| | Veri | Model | Satıcı tipi | Kaggle ağırlığı |
|---|---|---|---|---|
| A | yalnız Kaggle | eski HistGB (one-hot) | yok | 1 |
| B | yalnız Kaggle | eski HistGB (one-hot) | var | 1 |
| C | Kaggle + vehicles | büyük HistGB | var | 1 |
| D | Kaggle + vehicles | büyük HistGB | var | **5** (seçilen) |

Neden ağırlık 5? Eğitimde 168 bin vehicles ilanı var, Kaggle yalnızca 650. Ağırlık vermezsek model
vehicles dağılımına uyar; ağırlıkla model yarışma verisine daha çok uyar.

In [6]:
num_cols = ["year", "mileage", "cylinders"]
for df in (kaggle, vehicles):
    for c in num_cols:
        df[c] = df[c].astype(float)
cv = KFold(5, shuffle=True, random_state=42)
y_k = kaggle[TARGET].to_numpy(float)


def evaluate(name, make, cat_cols, use_vehicles, kaggle_weight=1.0):
    t0 = time.perf_counter()
    cols = num_cols + cat_cols
    oof = np.zeros(len(kaggle))
    for tr, te in cv.split(kaggle):
        parts, weights = [kaggle.iloc[tr]], [np.full(len(tr), kaggle_weight)]
        if use_vehicles:
            parts.append(vehicles)
            weights.append(np.ones(len(vehicles)))
        train = pd.concat(parts, ignore_index=True)
        old, models.CAT_COLS = models.CAT_COLS, cat_cols  # make_* bu listeyi okur
        model = make()
        models.CAT_COLS = old
        model.fit(train[cols], train[TARGET].astype(float),
                  reg__sample_weight=np.concatenate(weights))
        oof[te] = model.predict(kaggle.iloc[te][cols])
    row = {
        "MAE ($)": mean_absolute_error(y_k, oof),
        "R²": r2_score(y_k, oof),
        "Medyan % hata": np.median(np.abs(oof - y_k) / y_k) * 100,
        "Süre (sn)": time.perf_counter() - t0,
    }
    print(name, {k: round(v, 3) for k, v in row.items()}, flush=True)
    return row


def old_model():
    return make_hist_gb(**BEST_PARAMS)


no_seller = [c for c in CAT_COLS if c != "seller_type"]
results = {
    "A) eski model, yalnız Kaggle, satıcı tipi yok": evaluate("A", old_model, no_seller, False),
    "B) eski model, yalnız Kaggle, satıcı tipi var": evaluate("B", old_model, CAT_COLS, False),
    "C) Kaggle + vehicles, büyük model": evaluate("C", make_big_hist_gb, CAT_COLS, True),
    "D) C + Kaggle ağırlığı 5 (seçilen)": evaluate(
        "D", make_big_hist_gb, CAT_COLS, True, kaggle_weight=KAGGLE_WEIGHT),
}
res = pd.DataFrame(results).T
res["R²"] = res["R²"].map("{:.3f}".format)
res["Medyan % hata"] = res["Medyan % hata"].map("{:.1f}".format)
res

A {'MAE ($)': 6139.47, 'R²': 0.485, 'Medyan % hata': np.float64(27.147), 'Süre (sn)': 5.895}


B {'MAE ($)': 5904.135, 'R²': 0.507, 'Medyan % hata': np.float64(26.426), 'Süre (sn)': 5.816}


C {'MAE ($)': 4466.617, 'R²': 0.762, 'Medyan % hata': np.float64(19.19), 'Süre (sn)': 113.613}


D {'MAE ($)': 4345.471, 'R²': 0.792, 'Medyan % hata': np.float64(19.872), 'Süre (sn)': 86.072}


,MAE ($),R²,Medyan % hata,Süre (sn)
"A) eski model, yalnız Kaggle, satıcı tipi yok","6,139",0.485,27.1,6
"B) eski model, yalnız Kaggle, satıcı tipi var","5,904",0.507,26.4,6
"C) Kaggle + vehicles, büyük model","4,467",0.762,19.2,114
D) C + Kaggle ağırlığı 5 (seçilen),"4,345",0.792,19.9,86


In [7]:
mae = [r["MAE ($)"] for r in results.values()]
labels = ["A", "B", "C", "D"]
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#9aa5b1", "#9aa5b1", "#5b8def", "#1f4fd8"]
bars = ax.bar(labels, mae, color=colors)
ax.bar_label(bars, labels=[f"{m:,.0f} $" for m in mae], padding=3)
ax.set_ylabel("MAE ($), düşük daha iyi")
ax.set_title("Kaggle ilanlarında hata (5 katlı CV)")
ax.set_ylim(0, max(mae) * 1.15)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xticks(range(4), ["A\neski, Kaggle", "B\n+ satıcı tipi", "C\n+ vehicles", "D\n+ ağırlık 5"])
fig.tight_layout()
fig.savefig("docs/img/veri_karsilastirma.png", dpi=150)
plt.close(fig)
print("kaydedildi: docs/img/veri_karsilastirma.png")

kaydedildi: docs/img/veri_karsilastirma.png


**Yorum.** Hata, ek veriyle belirgin biçimde düşer. Satıcı tipi (B) tek başına küçük bir fark
yaratır; asıl kazanç vehicles verisinden gelir (C), Kaggle ağırlığı (D) ise modeli bizim
yarışma verimize biraz daha yaklaştırır.

## 5. Yeni modelde özellik önemi

Kaggle ilanlarının %80'i (ağırlık 5) ve tüm vehicles ile eğitilen model, kalan %20 Kaggle ilanında
**permutation importance** ile sınanır: bir sütunun değerleri karıştırılır, hata ne kadar artarsa
sütun o kadar önemlidir. `n_repeats=3`.

In [8]:
tr_k, va_k = train_test_split(kaggle, test_size=0.2, random_state=42)
train = pd.concat([tr_k, vehicles], ignore_index=True)
weights = np.concatenate([np.full(len(tr_k), KAGGLE_WEIGHT), np.ones(len(vehicles))])
model = make_big_hist_gb().fit(train[FEATURES], train[TARGET].astype(float),
                               reg__sample_weight=weights)
imp = permutation_importance(model, va_k[FEATURES], va_k[TARGET].astype(float), n_repeats=3,
                             random_state=42, scoring="neg_mean_absolute_error")
imp_df = (pd.DataFrame({"özellik": FEATURES, "MAE artışı ($)": imp.importances_mean,
                        "std": imp.importances_std})
          .sort_values("MAE artışı ($)", ascending=False).reset_index(drop=True))
imp_df

,özellik,MAE artışı ($),std
0,year,"4,237",318
1,mileage,"2,674",565
2,model,"2,453",482
3,brand,"1,186",90
4,cylinders,782,647
5,seller_type,565,179
6,title_status,248,47
7,condition,205,309
8,drive,102,95
9,body_type,-4,91


In [9]:
plot_df = imp_df.sort_values("MAE artışı ($)")
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(plot_df["özellik"], plot_df["MAE artışı ($)"], xerr=plot_df["std"], color="#1f4fd8")
ax.set_xlabel("Karıştırınca MAE artışı ($)")
ax.set_title("Özellik önemi (Kaggle doğrulama ilanları)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("docs/img/ozellik_onemi_v2.png", dpi=150)
plt.close(fig)
print("kaydedildi: docs/img/ozellik_onemi_v2.png")

kaydedildi: docs/img/ozellik_onemi_v2.png


**Yorum.** `year` ve `mileage` hâlâ en önemli özellikler; büyük veriyle `model` ve `brand`
artık anlamlı katkı veriyor (küçük veride model adı gürültüydü). `seller_type` listede görünür:
galeri ilanları aynı araç için bireysele göre daha pahalı.